In [ ]:
import sys
print(sys.executable)
import torch
print(torch.__version__)


In [ ]:
# If dfu_lab_eri_unet.py is in the same folder as this notebook:
from dfu_lab_eri_unet import TrainConfig, run_training

train_images_dir = r"C:\Users\User\Desktop\Foot Ulcer Segmentation Challenge\train\images"
train_labels_dir = r"C:\Users\User\Desktop\Foot Ulcer Segmentation Challenge\train\labels"
val_images_dir   = r"C:\Users\User\Desktop\Foot Ulcer Segmentation Challenge\validation\images"
val_labels_dir   = r"C:\Users\User\Desktop\Foot Ulcer Segmentation Challenge\validation\labels"

cfg = TrainConfig(
    img_size=(512, 512),
    batch_size=6,        # lower if GPU memory issues: 2 or 4
    num_workers=2,       # Windows: 0 or 2 is often stable
    lr=2e-4,
    epochs=50,
    lam_grad=0.05,
    base_channels=32,
)

ckpt_path = run_training(
    train_images_dir, train_labels_dir,
    val_images_dir, val_labels_dir,
    cfg,
    save_path="unet_lab_eri_best.pt"
)
print("Saved:", ckpt_path)


In [ ]:
import os, glob
import numpy as np
import cv2
import matplotlib.pyplot as plt

from predict_unet_lab_eri import load_checkpoint_model, read_rgb, predict_mask_single


In [ ]:
ckpt_path = r"c:\Users\User\Desktop\Foot Ulcer Segmentation Challenge\unet_lab_eri_best.pt"
test_dir  = r"c:\Users\User\Desktop\Foot Ulcer Segmentation Challenge\test\images"

model, mean, std, cfg, device = load_checkpoint_model(ckpt_path)
print("device:", device)
print("cfg base_channels:", cfg.get("base_channels", None))

# Match what your predict script supports (same extensions)
exts = ("*.png","*.jpg","*.jpeg","*.bmp","*.tif","*.tiff","*.PNG","*.JPG","*.JPEG","*.BMP","*.TIF","*.TIFF")
img_paths = []
for e in exts:
    img_paths.extend(glob.glob(os.path.join(test_dir, e)))
img_paths = sorted(img_paths)

print("Found test images:", len(img_paths))
img_path = img_paths[1]   # change index to pick another
print("Using:", img_path)

# read original
rgb = read_rgb(img_path)  # RGB uint8

# predict
prob, mask = predict_mask_single(
    model=model,
    rgb_uint8=rgb,
    mean=mean,
    std=std,
    size=(512, 512),
    device=device,
    thr=0.5,
)

# --- make probability overlay (heatmap blended with image) ---
prob8 = np.clip(prob * 255.0, 0, 255).astype(np.uint8)              # 0..255
heat_bgr = cv2.applyColorMap(prob8, cv2.COLORMAP_JET)               # BGR heatmap
heat_rgb = cv2.cvtColor(heat_bgr, cv2.COLOR_BGR2RGB)

alpha = 0.45
rgb_resized = cv2.resize(rgb, (512, 512), interpolation=cv2.INTER_LINEAR)

prob_overlay = cv2.addWeighted(rgb_resized, 1 - alpha, heat_rgb, alpha, 0)

# mask is already 0/255 at 512x512; just ensure shape
mask_show = mask

# --- plot ---
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(rgb_resized)
plt.title("Original (resized)")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(prob_overlay)
plt.title("Probability overlay (heatmap)")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(mask_show, cmap="gray")
plt.title("Binary mask (thr=0.5)")
plt.axis("off")

plt.show()

print("prob min/max:", float(prob.min()), float(prob.max()))
